## Ch11. Training Deep Neural Networks

복잡한 문제를 해결하기 위해서 심층 신경망(DNN)이 필요하다.

- DNN 훈련 네 가지 문제 -> 해결책
1. 그래디언트 소실/폭주 -> 스마트한 가중치 초기화, 더 나은 활성화 함수, 배치 정규화, 층 정규화, 그래디언트 클리핑
2. 훈련 데이터 부족 -> 전이 학습, 비지도 사전훈련
3. 느린 훈련 -> 다양한 옵티마이저, 학습률 스케줄링
4. 과적합 리스크 -> l1/l2 규제, 드롭아웃, 몬테가를로 드롭아웃, 맥스-노름 규제

#### 그래디언트 소실/폭주
역전파의 2단계는 출력층에서 입력층 방향으로 오차 그래디언트를 전파하며, 모든 파라미터에 대한 비용 함수의 그래디언트를 구한 뒤
경사 하강법으로 각 파라미터를 업데이트하는 과정이다.

##### 불안정한 그래디언트
- 그래디언트 소실: 아래쪽으로 갈수록 그래디언트가 점점 작아짐, 하위 층의 가중치가 거의 변하지 않아 훈련이 좋은해로 수렴하지 못함
- 그래디언트 폭주: 그래디언트가 점점 커짐, 층들이 엄청나게 큰 가중치 업데이트를 받아 알고리즘이 발산

##### 불안정함의 원인
시그모이드(로지스틱) 활성화 함수 + 가중치 초기화(평균0, 표준편차1인 정규분포)
1. 이 조합에서 각 층 출력의 분산이 입력의 분산보다 훨씬 커진다.
2. 네트워크를 따라 앞으로 갈수록 분산이 층마다 계속 증가한다.
3. 결국 상위 층에서 활성화 함수가 포화된다. -> 그 구간에서 도함수가 0에 극히 가까워진다.
4. 시그모이드의 평균이 0이 아니라 0.5라는 점이 포화를 더 악화시킨다.

초기화 방식과 활성화 함수를 바꾸는 방향으로 해결한다.

#### 해결책: 초기화 방식

아이디어: 순전파 - 신호가 소멸하거나 폭주·포화되지 않아야 함, 역전파 - 마찬가지로 그래디언트가 소멸/폭주하지 않아야 함
두 조건:
1) 각 층 출력의 분산 = 입력의 분산
2) 역방향으로 층을 통과하기 전후의 그래디언트 분산이 같을 것

##### Glorot 초기화
fan_avg = (fan_in + fan_out)/2 일 때, 가중치를 다음 중 하나로 무작위 초기화한다.
1) 정규분포: 평균 0, 분산 σ² = 1 / fan_avg
2) 균등분포: −r ~ +r, r = √(3 / fan_avg)

##### LeCun 초기화
fan_in = fan_out, 훈련을 크게 가속하며 딥러닝 성공의 핵심 트릭 중 하나이다.

##### 활성화 함수별 초기화
- Xavier Glorot
적용 활성화 함수: 없음, tanh, sigmoid, softmax
- Kaiming He
적용 활성화 함수: ReLU, Leaky ReLU, ELU, GELU, Swish, Mish, SwiGLU, ReLU^2
- Yann LeCun
적용 활성화 함수: SELU

##### PyTorch에서의 초기화
nn.Linear는 Kaiming 균등 초기화에서 가중치를 √6배 줄인 상태로 초기화하며, 편향도 무작위로 초기화하며
이 스케일은 어떤 일반적인 활성화 함수에도 최적이 아니다.

- 방법1: 직접 스케일 조정
- 방법2: torch.nn.init 함수 사용
- 방법3: 모델 전체에 일괄 적용

In [ ]:
# 방법1: 직접 스케일 조정
import torch
import torch.nn as nn

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5
torch.zero_(layer.bias.data)

# 방법2: torch.nn.init 함수 사용
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

# 방법3: 모델 전체에 일괄 적용
def use_he_init(module):
	if isinstance(module, nn.Linear):
		nn.init.kaiming_uniform_(module.weight)
		nn.init.zeros_(module.bias)

model = nn.Sequential(nn.Linear(50, 40), nn.ReLU(), nn.Linear(40, 1), nn.ReLU())
model.apply(use_he_init)

#### 해결책: 더 나은 활성화 함수
- ReLU가 좋은 이유: 양수 구간에서 포화되지 않고, 계산이 매우 빠르다.
- ReLU의 문제: 죽은 ReLU, 훈련 중 일부 뉴런이 죽어서 0 이외의 값을 출력하지 않게 되고, 특히 학습률이 클 때 네트워크 뉴런의 절반이 죽는 경우도 있다.
- 해결책: ReLU의 변종을 사용한다. 대표적으로 leaky ReLU가 있다.

#### Leaky ReLU
##### LeakyReLU_α(z) = max(αz, z)
- 하이퍼파라미터 α는 z < 0일 때의 기울기, 즉 함수가 얼마나 새는지를 결정
- 음수 구간에도 기울기가 있어서 절대 완전히 죽지 않는다.

##### 세 가지 변종
1) LeakyReLU: α 고정된 하이퍼파라미터 / 가장 기본형
2) RReLU: α 훈련 중 주어진 범위에서 무작위 선택, 테스트 시에는 평균값으로 고정 / 성능이 꽤 좋고 규제 효과가 있어 과적합 리스크 감소
3) PReLU: α를 훈련 중 학습(역전파로 갱신되는 파라미터) / 대규모 이미지 데이터셋에서는 ReLU보다 크게 우수하지만, 작은 데이터셋에서는 과적합 위험

문제는 z=0에서 기울기가 갑자기 바뀌는 매끄럽지 않은 함수이고, 이런 도함수의 불연속은 경사 하강법이 최적점 주변에서 튀며 수렴을 늦출 수 있다.

In [ ]:
# Leaky ReLU
alpha = 0.2
model = nn.Sequential(nn.Linear(50, 40), nn.LeakyReLU(negative_slope=alpha))
nn.init.kaiming_uniform_(model[0].weight, alpha, nonlinearity="leaky_relu")

#### ELU와 SELU
##### ELU
z < 0: ELU_α(z) = α(exp(z) − 1)
z ≥ 0: ELU_α(z) = z

- ReLU와의 주요 차이점
1) z<0에서 음수 값을 가짐: 평균 출력이 0에 가까워져 그래디언트 소실 완화
2) z<0에서도 기울기가 0이 아님: 죽은 뉴런 문제 방지
3) α = 1이면 z = 0 주변에서도 매끄러움: z=0 좌우로 덜 튀어서 경사 하강법이 빨라짐

- 지수 함수 때문에 ReLU 계열보다 계산이 느리다는 단점이 있다.

##### SELU
ELU의 스케일 버전

- 핵심 성질: 자기 정규화
- 밀집층만으로 쌓은 MLP에서 모든 은닉층이 SELU를 쓰면, 훈련 중 각 층의 출력이 평균0, 표준편차 1을 유지하는 경향이 있다.
이로써 그래디언트 소실/폭주 문제가 해결되고, 특이 심층 MLP에서 다른 활성화 함수보다 성능이 좋을 수 있다.
- 제약(자기 정규화 조건)이 매우 커서 기대와 달리 널리 쓰이지 못하고, 대부분의 작업에서 다른 활성화 함수가 꾸준히 더 좋은 성능을 보인다.

#### GELU, Swish, SwiGLU, Mish, ReLU²
공통 특징은 모두 ReLU의 매끄러운 변종이다. 이 함수들은 비볼록, 비단조이며 모든 지점에서 곡률이 있다.
복잡한 모양 덕분에 경사 하강법이 복잡한 패턴을 더 잘 맞출 수 있어, 특히 복잡한 작업에서 성능이 좋다.

##### GELU
- GELU(z) = z · Φ(z)
- 지금까지 본 함수들보다 성능이 좋은 경우가 많지만, 계산량이 더 많다.
- 근사식: GELU(z) ≈ z · σ(1.702z) (σ는 시그모이드). 훨씬 빠르다.

##### SiLU
- z · σ(z). GELU 논문에서 함께 제안되었으나 저자들의 테스트에서는 GELU에 밀림

##### Swish
- Swish_β(z) = z · σ(βz). GELU ≈ β = 1.702인 Swish. 논문에서는 GELU를 포함한 모든 함수를 능가

##### SwiGLU
- SwiGLU(z) = Swish_β(z) ⊗ Linear(z)
- 입력이 Swish를 통과하고, 동시에 선형층을 병렬로 통과한 뒤, 두 출력을 원소별 곱한다. 현대 트랜스포머에서 매우 흔히 쓰인다.

##### Mish
- mish(z) = z · tanh(softplus(z)), softplus(z) = log(1 + exp(z))
- 저자의 실험에서 Swish, GELU까지 근소한 차이로 능가했다.

##### ReLU²
- ReLU²(z) = (max(0, z))²
- 2021년 구글 연구자들이 대형 트랜스포머를 개선하기 위한 자동 아키텍처 탐색에서 발견한 단순하고 효과적인 함수

##### 상황별 추천
- 대부분의 작업: ReLU
- 복잡한 작업: Swish
- 약간 더 나은 성능이 필요: Mish, SwiGLU
- 실행 지연 시간 중시: leaky ReLU, 복잡한 작업이면 PReLU 또는 ReLU²

#### 배치 정규화
- BN = 층 입력을 미니배치 평균·분산으로 표준화 → γ, β로 재조정
- 훈련 때는 배치 통계, 추론 때는 이동 평균으로 구한 통계를 사용
- 효과: 소실 완화, 큰 학습률, 초기화 둔감, 약한 규제
- 단점인 추론 지연은 BN을 이전 층에 융합해 제거할 수 있다.

In [ ]:
# PyTorch로 배치 정규화 구현
# 1. BN을 첫 층과 각 은닉층 활성화 뒤에 배치
model = nn.Sequential(
	nn.Flatten(),
	nn.BatchNorm1d(1 * 28 ** 28),
	nn.Linear(1 * 28 * 28, 300),
	nn.ReLU(),
	nn.BatchNorm1d(300),
	nn.Linear(300, 100),
	nn.ReLU(),
	nn.BatchNorm1d(100),
	nn.Linear(100, 10)
)

# BN층의 파라미터(역전파로 학습)
print(dict(model[1].named_parameters()).keys)
# BN층의 버퍼(역전파가 아니라 이동 평균으로 갱신)
print(dict(model[1].named_buffers()).keys())

In [ ]:
# 2. BN을 활성화 함수 앞에 배치
model = nn.Sequential(
	nn.Flatten(),
	nn.Linear(1 * 28 * 28, 300, bias=False),
	nn.BatchNorm1d(300),
	nn.ReLU(),
	nn.Linear(300, 100, bias=False),
	nn.BatchNorm1d(100),
	nn.ReLU(),
	nn.Linear(100, 10)
)

#### 배치 정규화 1D, 2D, 3D
- 입력 모양에 맞춰 1D(특성), 2D(이미지), 3D(3D 이미지) 중 고르고, 이미지에서는 채널별로 통계를 공유하는 BatchNorm2D가 일반적
- 시퀀스 입력은 permute(0, 2, 1)로 특성 차원을 가운데로 옮긴 뒤 BatchNormld에 넣는다.
- BN은 배치 크기에 민감하고 RNN 등에서 약해, layer-norm이 점점 대체하고 있다.

#### 층 정규화
- LN = 샘플별로, 특성 차원들에 걸쳐 평균, 분산을 구해 정규화(BN은 배치 차원 기준).
- 샘플마다 독립적이라 훈련, 추론 동작이 같고, 이동 평균이 필요 없으며, 배치 크기에 민감하지 않다.
- nn.LayerNorm(정규화할 마지막 차원들의 크기)로 사용하며, 비전에서는 보통 채널 포함 전체를 정규화한다.

In [ ]:
# 층 정규화
# 1. 채널별로 따로 정규화(마지막 두 차원)
inputs = torch.randn(32, 3, 100, 200) # 랜덤 RGB 이미지 배치
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs) # 마지막 두 차원에 대해 정규화
print(result)

# 직접 계산한 동일한 결과
means = inputs.mean(dim=[2, 3], keepdim=True)
vars_ = inputs.var(dim=[2, 3], keepdim=True, unbiased=False)
stds = torch.sqrt(vars_ + layer_norm.eps)
result = layer_norm.weight * (inputs - means) / stds + layer_norm.bias
print(result)

# 2. 모든 채널을 한꺼번에 정규화(대부분의 컴퓨터 비전 구조)
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs) # 마지막 세 차원에 대해 정규화

NameError: name 'torch' is not defined

#### 그래디언트 클리핑
- 폭주하는 그래디언트의 크기를 상한으로 제한하는 기법이며, RNN에서 특히 유용
- clip_grad_norm_()은 방향 보존, clip_grad_value_()는 성분별 제한(방향 변경)
- 위치는 loss.backward() 직후, optimizer.step() 직전에 호출
- backward()(그래디언트 계산) -> 클리핑 -> step()(업데이트)

In [ ]:
# 훈련 루프 예
n_epochs = 1000
for epoch in range(n_epochs):
	for X_batch, y_batch in train_loader:
		X_batch, y_batch = X_batch.to(device), y_batch.to(device)
		y_pred = model(X_batch)
		loss = loss_fn(y_pred, y_batch)
		loss.backward()
		nn.utils. clip_grad_norm_(model.parameters(), max_norm=1.0)
		optimizer.step()
		optimizer.zero_grad()

#### 사전훈련된 층 재사용
- 전이 학습: 아주 큰 심층 신경망을 처음부터 훈련하는 것은 일반적으로 좋은 생각이 아니다. 먼저 비슷한 작업을 수행하는 기존 신경망을 찾아보고, 있다면 맨 위쪽 층을 제외한 대부분의 층을 재사용한다. 이 기법이 전이 학습이다.
- 훈련 속도가 크게 빨라지고, 필요한 훈련 데이터가 훨씬 적어진다.
- 출력층은 교체, 하위 층은 재사용, 상위 층은 작업 유사도와 데이터량에 따라 결정한다.
- 흐름: 전부 동결 -> 상위 층 일부 해제(낮은 학습률) -> 안되면 상위 층 제거/교체 순으로 최적의 층 수를 탐색
- 입력의 저수준 특성이 비슷해야 효과가 있다.

In [ ]:
# PyTorch로 전이 학습
# 모델 A(8개 클래스 분류)
torch.manual_seed(42)

model_A = nn.Sequential(
	nn.Flatten(),
	nn.Linear(1 * 28 * 28, 100),
	nn.ReLU(),
	nn.Linear(100, 100),
	nn.ReLU(),
	nn.Linear(100, 100),
	nn.ReLU(),
	nn.Linear(100, 8)
)
[...] # 훈련하거나 사전훈련 가중치 로드

In [ ]:
# 층 재사용: 출력층만 제외하고 복사
import copy

torch.manual_seed(42)
reused_layers = copy.deepcopy(model_A[:-1])
model_B_on_A = nn.Sequential(
	*reused_layers,
	nn.Linear(100, 1) # 작업 B용 새 출력층
).to(device)

# 재사용 층 동결
for layer in model_B_on_A[:-1]:
	for param in layer.parameters():
		param.requires_grad = False

# 이진 분류에 맞게 손실과 지표 변경
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device0)
[...] # model_B_on_A 훈련

#### 비지도 사전훈련
- 레이블 없는 데이터로 먼저 표현을 학습 -> 하위 층 재사용 -> 레이블 데이터로 미세 조정
- 2006년 딥러닝 부활의 핵심이었으나, 그래디언트 문제가 풀린 뒤로는 지도 학습이 주류가 되었다.
- 오늘날에도 "레이블 적음 + 재사용 모델 없음 + 비레이블 많음"상황에서는 유효하며, 도구는 RBM에서 오토인코더/확산 모델로 바뀜
- 층별로 쌓던 greedy 방식은 옛 방식이고, 지금은 한 번에 훈련

#### 보조 작업으로 사전훈련
- 레이블이 부족하면 레이블을 쉽게 만들 수 있는 보조 작업으로 먼저 훈련하고, 하위 층을 재사용
- 대표 사례: 얼굴 동일인 판별(비전), 마스킹된 단어 예측(NLP)
- 데이터 수집 시 저작권과 개인정보 동의 문제를 반드시 확인해야 함
- 자기지도 학습은 LLM 훈련의 기반 개념

#### 더 빠른 옵티마이저
- 옵티마이저 교체는 초기화, 활성화 함수, 정규화, 전이 학습에 이은 다섯 번째 훈련 가속 방법이다.
- 모멘텀, 네스테로프 가속 경사, AdaGrad, RMSProp, Adam과 그 변종들

#### 모멘텀
- 모멘텀 = 과거 그레이디언트를 누적한 속도 벡터로 가중치를 업데이트 (m ← βm − η∇J, θ ← θ + m)
- β = 0.9면 최대 약 10배 가속. 평탄 구간, 길쭉한 골짜기, 스케일이 다른 입력에서 효과적
- 오버슈팅과 진동이 생길 수 있어 마찰(β)이 필요
- Pytorch에서는 SGD(..., momentum=0.9) 한 줄이면 된다.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.05)

#### 네스테로프 가속 경사(NAG)
- NAG = 모멘텀에서 그래디언트를 "앞선 지점(θ + βm)"에서 측정하는 변형
- 모멘텀이 이미 가는 방향을 미리 내다보고 보정하므로 더 정확하고 진동이 적어 더 빠르게 수렴한다
- 비용 없이 옵션 하나(nesterov=True)로 켤 수 있어 기본으로 시도해 볼 만하다.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, nesterov=True, lr=0.05)

#### AdaGrad
- Adagrad = 그래디언트 제곱을 누적해서 그 제곱근으로 나눔 -> 파라미터마다 학습률이 자동으로 달라짐
- 가파른 차원의 스텝을 줄여 방향을 일찍 보정하고, η 튜닝 부담도 감소
- 단, 누적만 하기에 학습률이 계속 줄어 신경망에서는 조기 정지 -> 실전에서는 안씀

#### RMSProp
- RMSProp = AdaGrad에서 그래디언트 제곱의 누적을 지수 이동 평균으로 바꿔 학습률이 과도하게 줄어드는 문제를 해결
- 기본 α = 0.9로 대부분 충분

In [ ]:
optimizer = torch.optim.RMSprop(model.parameters(), alpha=0.9, lr=0.05)

#### Adam
- Adam = 모멘텀(1차 모멘트) + RMSProp(2차 모멘트) + 초기 편향 보정
- 기본값(β₁=0.9, β₂=0.999, η=0.001)으로 대부분 잘 작동
- 적응형 학습률이라 튜닝이 쉬워 현재 가장 널리 쓰이는 옵티마이저 중 하나

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.999), lr=0.05)

#### AdaMax
- ℓ∞ 노름 사용, 가끔 더 안정적
#### NAdam
- Adam + Nesterov
#### AdamW
- Adam + 올바른 가중치 감쇠

#### 학습률 스케줄링
- 학습률은 너무 높아도(발산, 후반 진동), 너무 낮아도(느림, 국소 최적점) 문제다
- 가장 일반적인 처방은 높게 시작해서 점차 낮추는 것

#### 지수 스케줄링
- 지수 스케줄링 = 매 에포크 학습률 X gamma (gamma < 1, 1에 가깝게)

In [ ]:
model = [...]
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
schedular = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

for epoch in range(n_epochs):
	for X_batch, y_batch in train_loader:
		[...]
	scheduler.step()

#### 코사인 어닐링
- 코사인 어닐링 = η_max에서 η_min까지 코사인 곡선으로 부드럽게 감소. 대부분의 구간에서 학습률이 높게 유지되어
지수 스케줄링보다 대체로 성능이 좋다.
- 단, 총 에포크 수(T_max)를 미리 정해야 하는 것이 약점이다.

In [ ]:
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=0.001)

#### 성능 스케줄링
- PyTorch의 ReduceLROnPlateau 스케줄러로 구현
- 훈련 중 지정한 지표를 추적
- 이 지표가 일정 기간 개선되지 않으면 학습률에 어떤 계수를 곱해 줄인다.
- 앞의 지수·코사인 스케줄링과 달리 미리 정해진 일정이 아니라 실제 성능에 반응하므로, 총 에포크 수를 미리 알 필요가 없다.

In [ ]:
[...]
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
	optimizer, mode="max", patience=2, factor=0.1
)

metric = torchmetrics.Accuracy(task="multiclass", numclasses=10).to(device)
for epoch in range(n_epochs):
	for X_batch, y_batch in train_loader:
		[...]
	val_metric = evaluate_tm(model, valid_loader, metric).item()
	scheduler.step(val_metric)

#### 학습룰 워밍업
- 워밍업 = 학습률을 거의 0에서 시작해 몇 에포크 동안 최대값까지 올리는 것. 민감한 모델이나 큰 배치에서 초반 불안정을 막는다.
- LinearLR(간단) 또는 LambdaLR(유연)로 구현하고, 워밍업 중에는 다른 스케줄러를 꺼야 한다.
- 훈련 도중 plateau 탈출용으로 학습률을 잠시 올리는 방법이 필요하며, 가장 쉬운 방법이 웜 리스타트 코사인 어닐링

In [ ]:
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
	optimizer, start_factor=0.1, end_factor=1.0, total_iters=3
)
warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(
	optimizer, lambda epoch: (min(epoch, 3) / 3) * (1.0 - 0.1) + 0.1
)
for epoch in range(n_epochs):
	warmup_scheduler.step()
	for X_batch, y_batch in train_loader:
		[...]
	if epoch >= 3:
		schedular.step(val_metric)

#### 웜 리스타트가 있는 코사인 어닐링
- 웜 리스타트 = 코사인 어닐링을 반복하며 학습률을 주기적으로 다시 올림. -> 국소 최적점 및 plateau 자동 탈출
- 짧게 시작해서 T_max를 두 배씩 늘리는 것이 권장 전략이며, T_0과 T_mult=2로 설정
- 어느 시점에 멈춰도 주기 끝 지점에서는 학습률이 낮아 비교적 좋은 모델을 얻기 쉽다.

#### 1cycle 스케줄링
- 1cycle = 학습률을 올렸다가 내리고, 마지막에 크게 떨어뜨리는 한 번의 큰 주기.
모멘텀은 그 반대로 움직인다.
- 적은 에포크로 더 좋은 성능에 도달하는 초수렴을 보였다.

#### 규제를 통한 과대적합 방지
- 파라미터가 많은 심층 신경망은 표현력이 크지만 과대적합에 취약하므로 규제가 필수

#### l1 및 l2 규제
- l1: 희소 모델(많은 가중치가 0) 유도
- l2: 가중치 크기를 제한

In [ ]:
# 일부 파라미터만 규제하는 방법
# 방법1: l2 손실 수동 계산
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
params_to_regularize = [
	param for name, param in model.named_parameters()
	if not "bias" in name and not "bn" in name
]
for epoch in range(n_epochs):
	for X_batch, y_batch in train_loader:
		[...]
		main_loss = loss_fn(y_pred, y_batch)
		l2_loss = sum(param.pow(2.0).sum() for param in params_to_regularize)
		loss = main_loss + 1e-4 * l2_loss
		[...]

# 방법 2: 파라미터 그룹
params_bias_and_bn = [
	param for name, param in model.named_parameters()
	if "bias" in name or "bn" in name
]
optimizer = torch.optim.SGD([
	{"params": params_to_regularize, "weight_decay": 1e-4},
	{"params": params_bias_and_bn}
], lr=0.05)

In [ ]:
# l1 규제
l1_loss = sum(param.abs().sum() for param in params_to_regularize)
loss = main_loss + 1e-4 * l1_loss

#### 드롭아웃
- 드롭아웃 = 훈련 중 뉴런을 확률 p로 무작위 비활성화해서 공동 적응을 막고 앙상블 효과를 내는 규제 기법. 보통 p = 0.1~0.5
- 훈련 중 (1-p)로 나눠 스케일을 보정하며, PyTorch의 nn.Dropout이 자동 처리
- train() / eval() 모드 전환이 필수이고, 훈련 손실은 드롭아웃을 끈 상태로 재측정해야 정확하다.
- 출력층은 드롭하지 않고, 상위 은닉층 위주로 적용하는 것이 효율적이다.

In [ ]:
model = nn.Sequential(
	nn.Flatten(),
	nn.Dropout(p=0.2), nn.Linear(1 * 28 * 28, 100), nn.ReLU(),
	nn.Dropout(p=0.2), nn.Linear(100, 100), nn.ReLU(),
	nn.Dropout(p=0.2), nn.Linear(100, 100), nn.ReLU(),
	nn.Dropout(p=0.2), nn.Linear(100, 10)
).to(device)

#### 몬테카를로 드롭아웃
- MC 드롭아웃 = 예측 시에도 드롭아웃을 켜고 N번 예측해 평균. 재훈련 불필요.
- 확률로 바꾼 뒤 평균을 내야 하며, 로짓 평균은 과신을 낳는다.
- 평균은 더 믿을 만한 확률, 표준편차는 불확실성 지표를 제공
- 샘플 수는 속도와 정확도의 트레이드오프이고, 훈련 단계에서는 일반 드롭아웃처럼 규제 효과도 있음.

In [ ]:
model.eval()
for module in model.modules():
	if isinstance(module, nn.Dropout):
		module.train()
X_new = [...]
X_new = X_new.to(device)

torch.manual_seed(42)
with torch.no_grad():
	X_new_repeated = X_new.repeat_interleave(100, dim=0)
	y_logits_all = model(X_new_repeated).reshape(3, 100, 10)
	y_probas_all = torch.nn.functional.softmax(y_logits_all, dim=-1)
	y_probas = y_probas_all.mean(dim=1)

y_std = y_probas_all.std(dim=1)

# 처음부터 MC 드롭아웃 모델을 훈련한다면
class McDropout(nn.Dropout):
	def forward(self, input):
		return F.dropout(input, self.p, training=True)

#### 맥스-노름 규제
- 뉴런별 입력 가중치의 l2노름 상한 r을 강제하는 규제
- r이 작을수록 규제가 강하고, 불안정한 그레이디언트 완화 효과도 있습니다.
- 편향은 제외하고, 층 종류에 맞게 dim을 설정

In [ ]:
def apply_max_norm(model, max_norm=2, epsilon=1e-8, dim=1):
	with torch.no_grad():
		for name, param in model.named_parameters():
			if 'bias' not in name:
				actual_norm = param.norm(p=2, dim=dim, keepdim=True)
				target_norm = torch.clamp(actual_norm, 0, max_norm)
				param *= target_norm / (epsilon + actual_norm)

#### Exercises

1. Glorot 초기화와 He 초기화가 해결하고자 하는 문제
-> 초기 가중치 스케일이 잘못돼서 층을 지날 때 신호가 사라지거나 폭주하는 문제를 층을 지나도 분산이 유지되도록 하여 해결한다.

2. 가중치를 모두 같은 값으로 초기화해도 괜찮은가?(그 값을 He 초기화로 뽑았더라도)
-> 안 된다. 대칭성 문제. 한 층의 모든 가중치가 같으면, 그 층의 모든 뉴런이 완전히 똑같은 계산을 한다.

3. 편향항을 0으로 초기화해도 괜찮은가?
-> 괜찮다. 가중치와 편향의 역할이 다르기 때문. 편향은 대칭을 깨는 역할이 필요 없다.

4. 이 장에서 논의한 각 활성화 함수는 어떤 경우에 사용하나?
- ReLU: 대부분의 작업의 기본값
- Leaky ReLU: 지연시간이 중요할 때, 죽은 ReLU가 문제일 때
- PReLU: 지연 시간이 중요하면서 복잡한 작업
- RReLU: 과대적합 문제일 때
- ELU: 수렴을 빠르게 하고 싶고 추론 속도는 덜 중요
- SELU: 순후 MLP이고 자기 정규화 조건을 모두 지킬 수 있을 때
- GELU: 복잡한 작업, 트랜스포머 계열
- Swish: 깊은 모델, 복잡한 작업의 기본값
- SwiGLU: 트랜스포머, 약간 더 나은 성능이 필요할 때
- Mish: Swish/GELU보다 근소하게 더 나은 성능을 원할 때
- ReLU²: 지연 시간이 중요하고 특히 희소 모델일 때

5. SGD 옵티마이저를 사용할 때 모멘텀 하이퍼파라미터를 1에 너무 가깝게 설정하면 어떤 일이 발생할 수 있나?
-> 최솟값을 지나치면 오래 진동하고, 심하면 발산한다.

6. 희소 모델을 만드는 세 가지 방법
- 훈련 후 작은 가중치 제거(가지치기)
- 훈련 중 강한 l1 규제
- 초기화 때 무작위 가중치를 작게 스케일

7. 드롭아웃은 학습 속도를 늦추나? 추론 속도를 늦추나? MC 드롭아웃은 어떤가
-> 드롭아웃은 훈련을 느리게 하고, 추론은 느리게 하지 않는다. MC 드롭아웃은 추론도 느리게 한다.

In [ ]:
# Exercises 8. CIFAR10_DNN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import copy

# 디바이스 설정 및 하이퍼파라미터
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

BATCH_SIZE = 128
EPOCHS = 40
PATIENCE = 10

# 데이터 로딩 및 전처리 (Standard Normalization)
# CIFAR-10 전체 평균 및 표준편차로 정규화
transform_standard = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_standard)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_standard)

# 45,000개 학습 / 5,000개 검증 데이터 분할
train_dataset, val_dataset = random_split(
    train_full, [45000, 5000], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


# 모델 정의 (Swish, BatchNorm, SELU, Alpha Dropout)

# Base Model: 20 Hidden Layers (He Normal Init + Swish)
class DeepMLPSwish(nn.Module):
    def __init__(self, in_features=32*32*3, hidden_units=100, num_layers=20, num_classes=10):
        super().__init__()
        layers = [nn.Flatten()]
        
        layers.append(nn.Linear(in_features, hidden_units))
        layers.append(nn.SiLU())  # Swish
        
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_units, hidden_units))
            layers.append(nn.SiLU())
            
        layers.append(nn.Linear(hidden_units, num_classes))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.net(x)


# Model with Batch Normalization
class DeepMLPBatchNorm(nn.Module):
    def __init__(self, in_features=32*32*3, hidden_units=100, num_layers=20, num_classes=10):
        super().__init__()
        layers = [nn.Flatten()]
        
        for i in range(num_layers):
            in_dim = in_features if i == 0 else hidden_units
            layers.append(nn.Linear(in_dim, hidden_units))
            layers.append(nn.BatchNorm1d(hidden_units))
            layers.append(nn.SiLU())
            
        layers.append(nn.Linear(hidden_units, num_classes))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.net(x)


# Self-Normalizing Network (LeCun Normal Init + SELU + Alpha Dropout)
class SelfNormalizingMLP(nn.Module):
    def __init__(self, in_features=32*32*3, hidden_units=100, num_layers=20, num_classes=10, use_alpha_dropout=False, p=0.05):
        super().__init__()
        layers = [nn.Flatten()]
        
        for i in range(num_layers):
            in_dim = in_features if i == 0 else hidden_units
            layers.append(nn.Linear(in_dim, hidden_units))
            layers.append(nn.SELU())
            if use_alpha_dropout:
                layers.append(nn.AlphaDropout(p=p))
                
        layers.append(nn.Linear(hidden_units, num_classes))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                # LeCun Normal Initialization
                nn.init.normal_(m.weight, mean=0.0, std=(1.0 / m.weight.shape[1]) ** 0.5)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.net(x)


# 학습 및 검증 루프 (Early Stopping 포함)
def train_model(model, train_loader, val_loader, optimizer, criterion, patience=10, max_epochs=40, scheduler=None):
    best_loss = float('inf')
    patience_counter = 0
    best_weights = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            # OneCycleLR은 step마다 업데이트
            if scheduler and isinstance(scheduler, optim.lr_scheduler.OneCycleLR):
                scheduler.step()
                
            train_loss += loss.item() * inputs.size(0)
            
        # Validation
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                _, preds = outputs.max(1)
                correct += preds.eq(targets).sum().item()
                total += targets.size(0)
                
        val_loss /= total
        val_acc = correct / total
        
        if scheduler and not isinstance(scheduler, optim.lr_scheduler.OneCycleLR):
            scheduler.step()
            
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            best_weights = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"   [Early Stopped] Epoch {epoch+1:02d} | Best Val Loss: {best_loss:.4f} | Val Acc: {val_acc:.4f}")
                break
                
    model.load_state_dict(best_weights)
    return model


def evaluate(model, data_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
    return correct / total


# MC Dropout 전용 평가 함수
def evaluate_mc_dropout(model, data_loader, n_samples=50):
    model.eval()
    # AlphaDropout 레이어만 train 모드로 전환하여 활성화
    for m in model.modules():
        if isinstance(m, nn.AlphaDropout):
            m.train()
            
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # n_samples 회 샘플링하여 확률 추정
            stacked_probs = torch.stack([
                torch.softmax(model(inputs), dim=1) for _ in range(n_samples)
            ], dim=0)
            
            mean_probs = stacked_probs.mean(dim=0)
            preds = mean_probs.argmax(dim=1)
            
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
            
    return correct / total


# 단계별 실험 실행
criterion = nn.CrossEntropyLoss()

print("\n--- 1. Base Model (Swish + He Init + NAdam) ---")
model_base = DeepMLPSwish().to(device)
optimizer_base = optim.NAdam(model_base.parameters(), lr=1e-3)
model_base = train_model(model_base, train_loader, val_loader, optimizer_base, criterion, patience=PATIENCE, max_epochs=EPOCHS)
print(f"Base Test Accuracy: {evaluate(model_base, test_loader) * 100:.2f}%")


print("\n--- 2. Batch Normalization Model ---")
model_bn = DeepMLPBatchNorm().to(device)
optimizer_bn = optim.NAdam(model_bn.parameters(), lr=2e-3)
model_bn = train_model(model_bn, train_loader, val_loader, optimizer_bn, criterion, patience=PATIENCE, max_epochs=EPOCHS)
print(f"BatchNorm Test Accuracy: {evaluate(model_bn, test_loader) * 100:.2f}%")


print("\n--- 3. Self-Normalizing Model (SELU + LeCun Init) ---")
model_selu = SelfNormalizingMLP().to(device)
optimizer_selu = optim.NAdam(model_selu.parameters(), lr=5e-4)
model_selu = train_model(model_selu, train_loader, val_loader, optimizer_selu, criterion, patience=PATIENCE, max_epochs=EPOCHS)
print(f"SELU Test Accuracy: {evaluate(model_selu, test_loader) * 100:.2f}%")


print("\n--- 4. Alpha Dropout & MC Dropout ---")
model_alpha = SelfNormalizingMLP(use_alpha_dropout=True, p=0.05).to(device)
optimizer_alpha = optim.NAdam(model_alpha.parameters(), lr=5e-4)
model_alpha = train_model(model_alpha, train_loader, val_loader, optimizer_alpha, criterion, patience=PATIENCE, max_epochs=EPOCHS)

acc_standard = evaluate(model_alpha, test_loader)
acc_mc = evaluate_mc_dropout(model_alpha, test_loader, n_samples=30)
print(f"Alpha Dropout Standard Test Accuracy: {acc_standard * 100:.2f}%")
print(f"MC Dropout Test Accuracy (30 Samples): {acc_mc * 100:.2f}%")


print("\n--- 5. 1cycle Scheduling (with BatchNorm) ---")
model_1cycle = DeepMLPBatchNorm().to(device)
optimizer_1cycle = optim.NAdam(model_1cycle.parameters(), lr=1e-3)

scheduler_1cycle = optim.lr_scheduler.OneCycleLR(
    optimizer_1cycle,
    max_lr=3e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.3
)

model_1cycle = train_model(
    model_1cycle, train_loader, val_loader, optimizer_1cycle, criterion, 
    patience=EPOCHS, max_epochs=EPOCHS, scheduler=scheduler_1cycle
)
print(f"1cycle Scheduling Test Accuracy: {evaluate(model_1cycle, test_loader) * 100:.2f}%")

Using device: cpu


100%|██████████| 170M/170M [28:25<00:00, 100kB/s]  



--- 1. Base Model (Swish + He Init + NAdam) ---
   [Early Stopped] Epoch 23 | Best Val Loss: 1.4819 | Val Acc: 0.4880
Base Test Accuracy: 49.81%

--- 2. Batch Normalization Model ---
   [Early Stopped] Epoch 19 | Best Val Loss: 1.4100 | Val Acc: 0.5102
BatchNorm Test Accuracy: 49.84%

--- 3. Self-Normalizing Model (SELU + LeCun Init) ---
   [Early Stopped] Epoch 18 | Best Val Loss: 1.4568 | Val Acc: 0.5018
SELU Test Accuracy: 49.41%

--- 4. Alpha Dropout & MC Dropout ---
   [Early Stopped] Epoch 17 | Best Val Loss: 1.7244 | Val Acc: 0.4998
Alpha Dropout Standard Test Accuracy: 46.95%
MC Dropout Test Accuracy (30 Samples): 46.62%

--- 5. 1cycle Scheduling (with BatchNorm) ---
1cycle Scheduling Test Accuracy: 52.26%
